# L7 — MCP (Model Context Protocol)

> **Un protocolo estándar para que cualquier LLM pueda hablar con cualquier sistema externo.** En lugar de integraciones custom, un contrato común.

## ¿Qué vas a aprender?

- Qué es MCP y por qué importa (estandarización vs integraciones custom)
- Las tres capacidades de un servidor MCP: **resources, tools, prompts**
- Cómo construir un servidor MCP con `FastMCP`
- Cómo construir un cliente MCP que conecte a Claude
- El patrón de **discovery**: el cliente descubre las capacidades en runtime

## ¿Qué es MCP?

MCP es un **protocolo de comunicación estándar** que define cómo un LLM puede interactuar con sistemas externos. No es un almacén de contexto ni una librería — es un **contrato**, igual que REST define cómo se comunican aplicaciones web.

**Antes de MCP**, cada integración entre un modelo y un sistema externo era custom: tool use propio, formato propio, cliente propio.

**Con MCP**, cualquier cliente compatible (Claude, Cursor, tu propio agente...) puede conectarse a cualquier servidor MCP **sin código adicional**.

## Arquitectura

```
┌─────────────┐        MCP Protocol        ┌─────────────────┐
│  MCP Client │ ◄────────────────────────► │   MCP Server    │
│  (el LLM)   │                            │ (tu sistema)    │
└─────────────┘                            └─────────────────┘
```

| Componente | Qué es | Quién lo construye |
|------------|--------|---------------------|
| **MCP Client** | El modelo o agente que consume capacidades | Anthropic (Claude), Cursor, tu propio código |
| **MCP Server** | El sistema que expone capacidades | **Tú** (lo que vamos a construir aquí) |
| **MCP Protocol** | El contrato entre ambos | Estándar abierto (publicado por Anthropic) |

## Qué puede exponer un MCP Server

Tres tipos de capacidades:

### Resources — datos que el modelo puede leer (como un GET)
- Contenido de un archivo de un repositorio
- Documentación de un proyecto
- Resultado de una query
- Logs de un servicio

### Tools — acciones que el modelo puede ejecutar (como un POST)
- Crear un ticket en Jira
- Hacer un commit en un repositorio
- Enviar una notificación
- Ejecutar un script

### Prompts — plantillas reutilizables que el servidor expone al cliente
- *"Analiza este PR siguiendo nuestros estándares"*
- *"Genera el release note de esta versión"*

(En este notebook construimos **resources + tools**, no prompts.)

## El flujo completo

```
1. El cliente (LLM) se conecta al servidor MCP
2. El cliente DESCUBRE qué resources y tools están disponibles
3. El usuario hace una petición al LLM
4. El LLM decide qué resources leer o qué tools invocar
5. El servidor ejecuta la acción y devuelve el resultado
6. El LLM genera la respuesta con ese contexto
```

**El paso 2 es clave:** el cliente descubre las capacidades automáticamente. No tienes que hardcodear en el prompt qué tools existen — el servidor las describe y el cliente las entiende.

## Qué construimos en este nivel

Un sistema completo:

- **Servidor MCP**: expone documentación técnica + base de datos SQLite de tickets
- **Cliente MCP**: descubre las capacidades + usa Claude para consumirlas

> **Requisitos:** `pip install anthropic mcp python-dotenv`

---
# PARTE 1 — El servidor (`server.py`)

> **Cómo se ejecuta un servidor MCP:** normalmente como **proceso separado**. El cliente lo lanza como subproceso vía `stdio` y se comunica con él por stdin/stdout en formato JSON-RPC.

En este notebook vamos a:
1. **Estudiar el código del servidor** celda a celda
2. **Escribirlo a disco** (cell de "guardar a server.py") para que el cliente lo pueda lanzar como subproceso

> **Importante:** las celdas de código del servidor **definen** funciones pero **no arrancan el servidor** dentro del notebook (eso bloquearía el kernel). El servidor real se arranca cuando el cliente lo invoca.

## Setup del servidor

Imports básicos: `os`, `glob`, `sqlite3`, `datetime`, y `FastMCP` del SDK de MCP.

In [ ]:
"""
L7 — MCP Server: expone documentación técnica via Model Context Protocol
========================================================================
Un servidor MCP que cualquier cliente compatible puede conectar y usar.
No sabe nada del cliente — solo expone capacidades y las ejecuta cuando
se le pide.

Expone:
    Resources:
        docs://list            → lista los documentos disponibles
        docs://{filename}      → lee el contenido de un documento

    Tools:
        search_docs            → busca un término en toda la documentación
        create_ticket_draft    → genera un borrador de ticket de soporte
        get_open_tickets       → consulta tickets abiertos en la base de datos
        save_ticket            → guarda un ticket real en la base de datos
        close_ticket           → cierra un ticket con su resolución

Base de datos: SQLite local (tickets.db).
El servidor la crea y la puebla con datos de ejemplo al arrancar.
El cliente no sabe que existe una base de datos — solo ve las tools.

Transporte: stdio (el cliente arranca este script como subproceso y se
comunica por stdin/stdout). Es el transporte estándar para servidores MCP
locales — así funciona la integración con Claude Desktop, Cursor, etc.

Requisitos:
    pip install mcp
"""

import os
import glob
import sqlite3
from datetime import datetime
from mcp.server.fastmcp import FastMCP

## Configuración del servidor

Constantes (rutas a docs y a la BD) y la **instancia de FastMCP**. El nombre `"support-docs"` identifica al servidor en el ecosistema MCP — los clientes lo usan para mostrar qué servidores están conectados.

In [ ]:
# ─────────────────────────────────────────────
# Configuración
# ─────────────────────────────────────────────

DOCS_PATH = "./docs"
DB_PATH   = "./tickets.db"

# El nombre identifica al servidor en el ecosistema MCP.
# Los clientes lo usan para mostrar qué servidores están conectados.
mcp = FastMCP("support-docs")

## Base de datos — SQLite

El servidor gestiona una BD SQLite **completamente transparente para el cliente**. Desde fuera solo se ven tools con nombres semánticos (`get_open_tickets`, `save_ticket`...).

`init_db` crea la tabla y la rellena con tickets de ejemplo si está vacía. `get_db` abre una conexión nueva por llamada (SQLite no es thread-safe con conexión compartida).

In [ ]:
# ─────────────────────────────────────────────
# Base de datos — SQLite
#
# El servidor gestiona la BD de forma completamente transparente
# para el cliente. Desde fuera solo se ven tools con nombres semánticos.
# Mañana podrías migrar a PostgreSQL cambiando solo este bloque.
# ─────────────────────────────────────────────

def get_db() -> sqlite3.Connection:
    """
    Abre una conexión a la base de datos.
    Creamos una conexión nueva por llamada para evitar problemas
    de concurrencia — SQLite no es thread-safe con una sola conexión compartida.
    """
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row  # permite acceder a columnas por nombre: row["title"]
    return conn


def init_db() -> None:
    """
    Crea la tabla de tickets y la puebla con datos de ejemplo.
    Si la BD ya existe, no hace nada (IF NOT EXISTS).
    Se llama al arrancar el servidor — el cliente nunca la invoca directamente.
    """
    conn = get_db()
    conn.execute("""
        CREATE TABLE IF NOT EXISTS tickets (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            title       TEXT    NOT NULL,
            description TEXT    NOT NULL,
            severity    TEXT    NOT NULL DEFAULT 'P3',
            service     TEXT,
            status      TEXT    NOT NULL DEFAULT 'open',
            resolution  TEXT,
            created_at  TEXT    NOT NULL,
            closed_at   TEXT
        )
    """)

    # Insertar datos de ejemplo solo si la tabla está vacía
    count = conn.execute("SELECT COUNT(*) FROM tickets").fetchone()[0]
    if count == 0:
        sample_tickets = [
            ("Auth service returning 500 on login",
             "Users cannot log in since 09:15 UTC. Database connection timeouts in logs.",
             "P1", "auth"),
            ("Email confirmation shows old address",
             "After updating email, the confirmation message shows the previous address.",
             "P4", "frontend"),
            ("API gateway rejecting requests with 503",
             "Circuit breaker open on payments upstream. Started after last deploy.",
             "P2", "api_gateway"),
            ("DB connection pool exhausted during peak hours",
             "pg_stat_activity shows 200+ idle connections. Service restarts help temporarily.",
             "P2", "database"),
            ("Rate limiting incorrectly applied to enterprise plan",
             "Enterprise customer receiving 429 errors despite unlimited plan.",
             "P3", "api_gateway"),
        ]
        now = datetime.utcnow().isoformat()
        conn.executemany(
            "INSERT INTO tickets (title, description, severity, service, created_at) VALUES (?,?,?,?,?)",
            [(t[0], t[1], t[2], t[3], now) for t in sample_tickets],
        )

    conn.commit()
    conn.close()

## Resources — datos que el cliente puede leer

Dos resources expuestos por URI:

- **`docs://list`** → lista de documentos disponibles
- **`docs://{filename}`** → contenido de un documento concreto

Los decoradores `@mcp.resource(...)` registran la función como resource consultable por el cliente.

> **Nota de seguridad:** `os.path.basename(filename)` previene path traversal (que el cliente pida `../../etc/passwd`).

In [ ]:
# ─────────────────────────────────────────────
# Resources — datos que el cliente puede leer
#
# Un resource es contenido que el modelo puede consultar, como un GET en REST.
# El cliente lo pide por URI; el servidor devuelve el contenido.
# ─────────────────────────────────────────────

@mcp.resource("docs://list")
def list_documents() -> str:
    """
    Lista todos los documentos disponibles en el servidor.
    El cliente puede leer este resource para saber qué documentos existen
    antes de pedir uno concreto.
    """
    pattern = os.path.join(DOCS_PATH, "*.md")
    files   = sorted(glob.glob(pattern))

    if not files:
        return "No hay documentos disponibles."

    names = [os.path.basename(f) for f in files]
    return "\n".join(names)


@mcp.resource("docs://{filename}")
def read_document(filename: str) -> str:
    """
    Lee el contenido completo de un documento.
    El cliente construye la URI con el nombre del fichero:
        docs://authentication.md
        docs://database.md
        docs://api_gateway.md
    """
    # Evitar path traversal: el fichero solo puede estar en DOCS_PATH
    safe_name = os.path.basename(filename)
    filepath  = os.path.join(DOCS_PATH, safe_name)

    if not os.path.exists(filepath):
        return f"Documento '{safe_name}' no encontrado. Usa docs://list para ver los disponibles."

    with open(filepath, "r", encoding="utf-8") as f:
        return f.read()

## Tools — `search_docs` y `create_ticket_draft`

`@mcp.tool()` registra la función como tool invocable. **FastMCP genera automáticamente el JSON schema** a partir de los type hints.

- **`search_docs`**: busca un término en todos los `.md` y devuelve fragmentos con su fuente
- **`create_ticket_draft`**: genera un borrador de ticket formateado (sin persistirlo)

`_sla` es un helper privado (no se expone como tool).

In [ ]:
# ─────────────────────────────────────────────
# Tools — acciones que el cliente puede ejecutar
#
# Una tool es una operación, como un POST en REST.
# El cliente (o el LLM) decide cuándo invocarla y con qué parámetros.
# FastMCP genera el JSON schema de cada tool a partir de los type hints.
# ─────────────────────────────────────────────

@mcp.tool()
def search_docs(query: str) -> str:
    """
    Busca un término en toda la documentación y devuelve los fragmentos
    relevantes con el nombre del documento de origen.

    Útil cuando no sabes en qué documento está la información — el LLM
    puede buscar primero y luego leer el documento completo si necesita más.
    """
    pattern = os.path.join(DOCS_PATH, "*.md")
    files   = sorted(glob.glob(pattern))
    results = []

    query_lower = query.lower()

    for filepath in files:
        filename = os.path.basename(filepath)
        with open(filepath, "r", encoding="utf-8") as f:
            lines = f.readlines()

        for i, line in enumerate(lines):
            if query_lower in line.lower():
                # Devolvemos la línea con algo de contexto (±1 línea)
                start   = max(0, i - 1)
                end     = min(len(lines), i + 2)
                excerpt = "".join(lines[start:end]).strip()
                results.append(f"[{filename}]\n{excerpt}")

    if not results:
        return f"No se encontraron resultados para '{query}'."

    return "\n\n---\n\n".join(results)


@mcp.tool()
def create_ticket_draft(title: str, description: str, severity: str = "P3") -> str:
    """
    Genera un borrador de ticket de soporte estructurado.
    Devuelve el ticket formateado listo para copiar al sistema de tickets.

    severity: P1 (crítico) | P2 (alto) | P3 (medio) | P4 (bajo)
    """
    valid_severities = {"P1", "P2", "P3", "P4"}
    if severity not in valid_severities:
        severity = "P3"

    # En producción aquí iría una llamada real a Jira, Linear, etc.
    # El servidor MCP abstrae esa integración — el cliente no sabe
    # si esto escribe en Jira, en una base de datos o en un fichero.
    draft = f"""
BORRADOR DE TICKET
==================
Título:      {title}
Severidad:   {severity}
Estado:      Abierto
Asignado a:  Sin asignar

Descripción:
{description}

Pasos siguientes:
- [ ] Verificar en entorno de staging
- [ ] Revisar logs del servicio afectado
- [ ] Escalar si no hay resolución en {_sla(severity)}
""".strip()

    return draft


def _sla(severity: str) -> str:
    """Tiempo máximo de resolución según severidad."""
    return {"P1": "1 hora", "P2": "4 horas", "P3": "24 horas", "P4": "72 horas"}.get(severity, "24 horas")

## Tools de base de datos

Tres tools que tocan la BD SQLite:

- **`get_open_tickets`**: lectura — lista tickets abiertos con filtros opcionales
- **`save_ticket`**: escritura — crea un ticket nuevo
- **`close_ticket`**: actualización — cierra un ticket con su resolución

El cliente **no sabe** que hay una BD detrás — solo ve las tools.

In [ ]:
# ─────────────────────────────────────────────
# Tools — base de datos
# ─────────────────────────────────────────────

@mcp.tool()
def get_open_tickets(service: str = "", severity: str = "") -> str:
    """
    Devuelve los tickets abiertos de la base de datos.
    Se puede filtrar por servicio (auth, database, api_gateway, frontend)
    y/o por severidad (P1, P2, P3, P4).
    Dejar vacío para ver todos los tickets abiertos.
    """
    conn  = get_db()
    query = "SELECT * FROM tickets WHERE status = 'open'"
    params: list = []

    if service:
        query += " AND service = ?"
        params.append(service)
    if severity:
        query += " AND severity = ?"
        params.append(severity)

    query += " ORDER BY severity, created_at"

    rows = conn.execute(query, params).fetchall()
    conn.close()

    if not rows:
        return "No hay tickets abiertos con esos filtros."

    lines = []
    for row in rows:
        lines.append(
            f"[#{row['id']}] {row['severity']} | {row['service'] or 'N/A'} | {row['title']}\n"
            f"  {row['description'][:100]}..."
        )
    return "\n\n".join(lines)


@mcp.tool()
def save_ticket(title: str, description: str, severity: str = "P3", service: str = "") -> str:
    """
    Guarda un nuevo ticket en la base de datos y devuelve su ID.
    A diferencia de create_ticket_draft (que solo genera texto),
    esta tool persiste el ticket para que otros puedan consultarlo.

    severity: P1 | P2 | P3 | P4
    service:  auth | database | api_gateway | frontend | (vacío si no aplica)
    """
    valid_severities = {"P1", "P2", "P3", "P4"}
    if severity not in valid_severities:
        severity = "P3"

    conn = get_db()
    cursor = conn.execute(
        "INSERT INTO tickets (title, description, severity, service, created_at) VALUES (?,?,?,?,?)",
        (title, description, severity, service or None, datetime.utcnow().isoformat()),
    )
    ticket_id = cursor.lastrowid
    conn.commit()
    conn.close()

    return f"Ticket #{ticket_id} creado correctamente.\nTítulo: {title}\nSeveridad: {severity} — SLA: {_sla(severity)}"


@mcp.tool()
def close_ticket(ticket_id: int, resolution: str) -> str:
    """
    Cierra un ticket existente con la resolución aplicada.
    Devuelve error si el ticket no existe o ya está cerrado.
    """
    conn = get_db()
    row  = conn.execute("SELECT status FROM tickets WHERE id = ?", (ticket_id,)).fetchone()

    if not row:
        conn.close()
        return f"Ticket #{ticket_id} no encontrado."

    if row["status"] == "closed":
        conn.close()
        return f"Ticket #{ticket_id} ya está cerrado."

    conn.execute(
        "UPDATE tickets SET status = 'closed', resolution = ?, closed_at = ? WHERE id = ?",
        (resolution, datetime.utcnow().isoformat(), ticket_id),
    )
    conn.commit()
    conn.close()

    return f"Ticket #{ticket_id} cerrado.\nResolución: {resolution}"

## Arranque del servidor

`init_db()` se ejecuta al cargar el módulo para que la BD exista antes de aceptar conexiones.

> El `mcp.run()` original (que arrancaría el servidor con transporte stdio) **se ha omitido aquí** porque bloquearía el kernel del notebook. El servidor se arranca como subproceso desde el cliente.

In [ ]:
# ─────────────────────────────────────────────
# Arranque
# ─────────────────────────────────────────────

# Inicializar la BD antes de que el servidor empiece a aceptar conexiones.
# Si tickets.db no existe, se crea aquí con los datos de ejemplo.
init_db()

## Guardar `server.py` a disco

> **Esta celda es necesaria** para que el cliente pueda lanzar el servidor como subproceso. Escribe el código completo (incluyendo `mcp.run()`) a `./server.py`.

Usamos **base64** porque el código del servidor contiene comillas triples (`"""`) que romperían cualquier string literal que intentemos usar.

In [ ]:
import base64

SERVER_CODE_B64 = 'IiIiCkw3IOKAlCBNQ1AgU2VydmVyOiBleHBvbmUgZG9jdW1lbnRhY2nDs24gdMOpY25pY2EgdmlhIE1vZGVsIENvbnRleHQgUHJvdG9jb2wKPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09ClVuIHNlcnZpZG9yIE1DUCBxdWUgY3VhbHF1aWVyIGNsaWVudGUgY29tcGF0aWJsZSBwdWVkZSBjb25lY3RhciB5IHVzYXIuCk5vIHNhYmUgbmFkYSBkZWwgY2xpZW50ZSDigJQgc29sbyBleHBvbmUgY2FwYWNpZGFkZXMgeSBsYXMgZWplY3V0YSBjdWFuZG8Kc2UgbGUgcGlkZS4KCkV4cG9uZToKICAgIFJlc291cmNlczoKICAgICAgICBkb2NzOi8vbGlzdCAgICAgICAgICAgIOKGkiBsaXN0YSBsb3MgZG9jdW1lbnRvcyBkaXNwb25pYmxlcwogICAgICAgIGRvY3M6Ly97ZmlsZW5hbWV9ICAgICAg4oaSIGxlZSBlbCBjb250ZW5pZG8gZGUgdW4gZG9jdW1lbnRvCgogICAgVG9vbHM6CiAgICAgICAgc2VhcmNoX2RvY3MgICAgICAgICAgICDihpIgYnVzY2EgdW4gdMOpcm1pbm8gZW4gdG9kYSBsYSBkb2N1bWVudGFjacOzbgogICAgICAgIGNyZWF0ZV90aWNrZXRfZHJhZnQgICAg4oaSIGdlbmVyYSB1biBib3JyYWRvciBkZSB0aWNrZXQgZGUgc29wb3J0ZQogICAgICAgIGdldF9vcGVuX3RpY2tldHMgICAgICAg4oaSIGNvbnN1bHRhIHRpY2tldHMgYWJpZXJ0b3MgZW4gbGEgYmFzZSBkZSBkYXRvcwogICAgICAgIHNhdmVfdGlja2V0ICAgICAgICAgICAg4oaSIGd1YXJkYSB1biB0aWNrZXQgcmVhbCBlbiBsYSBiYXNlIGRlIGRhdG9zCiAgICAgICAgY2xvc2VfdGlja2V0ICAgICAgICAgICDihpIgY2llcnJhIHVuIHRpY2tldCBjb24gc3UgcmVzb2x1Y2nDs24KCkJhc2UgZGUgZGF0b3M6IFNRTGl0ZSBsb2NhbCAodGlja2V0cy5kYikuCkVsIHNlcnZpZG9yIGxhIGNyZWEgeSBsYSBwdWVibGEgY29uIGRhdG9zIGRlIGVqZW1wbG8gYWwgYXJyYW5jYXIuCkVsIGNsaWVudGUgbm8gc2FiZSBxdWUgZXhpc3RlIHVuYSBiYXNlIGRlIGRhdG9zIOKAlCBzb2xvIHZlIGxhcyB0b29scy4KClRyYW5zcG9ydGU6IHN0ZGlvIChlbCBjbGllbnRlIGFycmFuY2EgZXN0ZSBzY3JpcHQgY29tbyBzdWJwcm9jZXNvIHkgc2UKY29tdW5pY2EgcG9yIHN0ZGluL3N0ZG91dCkuIEVzIGVsIHRyYW5zcG9ydGUgZXN0w6FuZGFyIHBhcmEgc2Vydmlkb3JlcyBNQ1AKbG9jYWxlcyDigJQgYXPDrSBmdW5jaW9uYSBsYSBpbnRlZ3JhY2nDs24gY29uIENsYXVkZSBEZXNrdG9wLCBDdXJzb3IsIGV0Yy4KClJlcXVpc2l0b3M6CiAgICBwaXAgaW5zdGFsbCBtY3AKIiIiCgppbXBvcnQgb3MKaW1wb3J0IGdsb2IKaW1wb3J0IHNxbGl0ZTMKZnJvbSBkYXRldGltZSBpbXBvcnQgZGF0ZXRpbWUKZnJvbSBtY3Auc2VydmVyLmZhc3RtY3AgaW1wb3J0IEZhc3RNQ1AKCgojIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAojIENvbmZpZ3VyYWNpw7NuCiMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACgpET0NTX1BBVEggPSAiLi9kb2NzIgpEQl9QQVRIICAgPSAiLi90aWNrZXRzLmRiIgoKIyBFbCBub21icmUgaWRlbnRpZmljYSBhbCBzZXJ2aWRvciBlbiBlbCBlY29zaXN0ZW1hIE1DUC4KIyBMb3MgY2xpZW50ZXMgbG8gdXNhbiBwYXJhIG1vc3RyYXIgcXXDqSBzZXJ2aWRvcmVzIGVzdMOhbiBjb25lY3RhZG9zLgptY3AgPSBGYXN0TUNQKCJzdXBwb3J0LWRvY3MiKQoKCiMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiMgQmFzZSBkZSBkYXRvcyDigJQgU1FMaXRlCiMKIyBFbCBzZXJ2aWRvciBnZXN0aW9uYSBsYSBCRCBkZSBmb3JtYSBjb21wbGV0YW1lbnRlIHRyYW5zcGFyZW50ZQojIHBhcmEgZWwgY2xpZW50ZS4gRGVzZGUgZnVlcmEgc29sbyBzZSB2ZW4gdG9vbHMgY29uIG5vbWJyZXMgc2Vtw6FudGljb3MuCiMgTWHDsWFuYSBwb2Ryw61hcyBtaWdyYXIgYSBQb3N0Z3JlU1FMIGNhbWJpYW5kbyBzb2xvIGVzdGUgYmxvcXVlLgojIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKZGVmIGdldF9kYigpIC0+IHNxbGl0ZTMuQ29ubmVjdGlvbjoKICAgICIiIgogICAgQWJyZSB1bmEgY29uZXhpw7NuIGEgbGEgYmFzZSBkZSBkYXRvcy4KICAgIENyZWFtb3MgdW5hIGNvbmV4acOzbiBudWV2YSBwb3IgbGxhbWFkYSBwYXJhIGV2aXRhciBwcm9ibGVtYXMKICAgIGRlIGNvbmN1cnJlbmNpYSDigJQgU1FMaXRlIG5vIGVzIHRocmVhZC1zYWZlIGNvbiB1bmEgc29sYSBjb25leGnDs24gY29tcGFydGlkYS4KICAgICIiIgogICAgY29ubiA9IHNxbGl0ZTMuY29ubmVjdChEQl9QQVRIKQogICAgY29ubi5yb3dfZmFjdG9yeSA9IHNxbGl0ZTMuUm93ICAjIHBlcm1pdGUgYWNjZWRlciBhIGNvbHVtbmFzIHBvciBub21icmU6IHJvd1sidGl0bGUiXQogICAgcmV0dXJuIGNvbm4KCgpkZWYgaW5pdF9kYigpIC0+IE5vbmU6CiAgICAiIiIKICAgIENyZWEgbGEgdGFibGEgZGUgdGlja2V0cyB5IGxhIHB1ZWJsYSBjb24gZGF0b3MgZGUgZWplbXBsby4KICAgIFNpIGxhIEJEIHlhIGV4aXN0ZSwgbm8gaGFjZSBuYWRhIChJRiBOT1QgRVhJU1RTKS4KICAgIFNlIGxsYW1hIGFsIGFycmFuY2FyIGVsIHNlcnZpZG9yIOKAlCBlbCBjbGllbnRlIG51bmNhIGxhIGludm9jYSBkaXJlY3RhbWVudGUuCiAgICAiIiIKICAgIGNvbm4gPSBnZXRfZGIoKQogICAgY29ubi5leGVjdXRlKCIiIgogICAgICAgIENSRUFURSBUQUJMRSBJRiBOT1QgRVhJU1RTIHRpY2tldHMgKAogICAgICAgICAgICBpZCAgICAgICAgICBJTlRFR0VSIFBSSU1BUlkgS0VZIEFVVE9JTkNSRU1FTlQsCiAgICAgICAgICAgIHRpdGxlICAgICAgIFRFWFQgICAgTk9UIE5VTEwsCiAgICAgICAgICAgIGRlc2NyaXB0aW9uIFRFWFQgICAgTk9UIE5VTEwsCiAgICAgICAgICAgIHNldmVyaXR5ICAgIFRFWFQgICAgTk9UIE5VTEwgREVGQVVMVCAnUDMnLAogICAgICAgICAgICBzZXJ2aWNlICAgICBURVhULAogICAgICAgICAgICBzdGF0dXMgICAgICBURVhUICAgIE5PVCBOVUxMIERFRkFVTFQgJ29wZW4nLAogICAgICAgICAgICByZXNvbHV0aW9uICBURVhULAogICAgICAgICAgICBjcmVhdGVkX2F0ICBURVhUICAgIE5PVCBOVUxMLAogICAgICAgICAgICBjbG9zZWRfYXQgICBURVhUCiAgICAgICAgKQogICAgIiIiKQoKICAgICMgSW5zZXJ0YXIgZGF0b3MgZGUgZWplbXBsbyBzb2xvIHNpIGxhIHRhYmxhIGVzdMOhIHZhY8OtYQogICAgY291bnQgPSBjb25uLmV4ZWN1dGUoIlNFTEVDVCBDT1VOVCgqKSBGUk9NIHRpY2tldHMiKS5mZXRjaG9uZSgpWzBdCiAgICBpZiBjb3VudCA9PSAwOgogICAgICAgIHNhbXBsZV90aWNrZXRzID0gWwogICAgICAgICAgICAoIkF1dGggc2VydmljZSByZXR1cm5pbmcgNTAwIG9uIGxvZ2luIiwKICAgICAgICAgICAgICJVc2VycyBjYW5ub3QgbG9nIGluIHNpbmNlIDA5OjE1IFVUQy4gRGF0YWJhc2UgY29ubmVjdGlvbiB0aW1lb3V0cyBpbiBsb2dzLiIsCiAgICAgICAgICAgICAiUDEiLCAiYXV0aCIpLAogICAgICAgICAgICAoIkVtYWlsIGNvbmZpcm1hdGlvbiBzaG93cyBvbGQgYWRkcmVzcyIsCiAgICAgICAgICAgICAiQWZ0ZXIgdXBkYXRpbmcgZW1haWwsIHRoZSBjb25maXJtYXRpb24gbWVzc2FnZSBzaG93cyB0aGUgcHJldmlvdXMgYWRkcmVzcy4iLAogICAgICAgICAgICAgIlA0IiwgImZyb250ZW5kIiksCiAgICAgICAgICAgICgiQVBJIGdhdGV3YXkgcmVqZWN0aW5nIHJlcXVlc3RzIHdpdGggNTAzIiwKICAgICAgICAgICAgICJDaXJjdWl0IGJyZWFrZXIgb3BlbiBvbiBwYXltZW50cyB1cHN0cmVhbS4gU3RhcnRlZCBhZnRlciBsYXN0IGRlcGxveS4iLAogICAgICAgICAgICAgIlAyIiwgImFwaV9nYXRld2F5IiksCiAgICAgICAgICAgICgiREIgY29ubmVjdGlvbiBwb29sIGV4aGF1c3RlZCBkdXJpbmcgcGVhayBob3VycyIsCiAgICAgICAgICAgICAicGdfc3RhdF9hY3Rpdml0eSBzaG93cyAyMDArIGlkbGUgY29ubmVjdGlvbnMuIFNlcnZpY2UgcmVzdGFydHMgaGVscCB0ZW1wb3JhcmlseS4iLAogICAgICAgICAgICAgIlAyIiwgImRhdGFiYXNlIiksCiAgICAgICAgICAgICgiUmF0ZSBsaW1pdGluZyBpbmNvcnJlY3RseSBhcHBsaWVkIHRvIGVudGVycHJpc2UgcGxhbiIsCiAgICAgICAgICAgICAiRW50ZXJwcmlzZSBjdXN0b21lciByZWNlaXZpbmcgNDI5IGVycm9ycyBkZXNwaXRlIHVubGltaXRlZCBwbGFuLiIsCiAgICAgICAgICAgICAiUDMiLCAiYXBpX2dhdGV3YXkiKSwKICAgICAgICBdCiAgICAgICAgbm93ID0gZGF0ZXRpbWUudXRjbm93KCkuaXNvZm9ybWF0KCkKICAgICAgICBjb25uLmV4ZWN1dGVtYW55KAogICAgICAgICAgICAiSU5TRVJUIElOVE8gdGlja2V0cyAodGl0bGUsIGRlc2NyaXB0aW9uLCBzZXZlcml0eSwgc2VydmljZSwgY3JlYXRlZF9hdCkgVkFMVUVTICg/LD8sPyw/LD8pIiwKICAgICAgICAgICAgWyh0WzBdLCB0WzFdLCB0WzJdLCB0WzNdLCBub3cpIGZvciB0IGluIHNhbXBsZV90aWNrZXRzXSwKICAgICAgICApCgogICAgY29ubi5jb21taXQoKQogICAgY29ubi5jbG9zZSgpCgoKIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKIyBSZXNvdXJjZXMg4oCUIGRhdG9zIHF1ZSBlbCBjbGllbnRlIHB1ZWRlIGxlZXIKIwojIFVuIHJlc291cmNlIGVzIGNvbnRlbmlkbyBxdWUgZWwgbW9kZWxvIHB1ZWRlIGNvbnN1bHRhciwgY29tbyB1biBHRVQgZW4gUkVTVC4KIyBFbCBjbGllbnRlIGxvIHBpZGUgcG9yIFVSSTsgZWwgc2Vydmlkb3IgZGV2dWVsdmUgZWwgY29udGVuaWRvLgojIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKQG1jcC5yZXNvdXJjZSgiZG9jczovL2xpc3QiKQpkZWYgbGlzdF9kb2N1bWVudHMoKSAtPiBzdHI6CiAgICAiIiIKICAgIExpc3RhIHRvZG9zIGxvcyBkb2N1bWVudG9zIGRpc3BvbmlibGVzIGVuIGVsIHNlcnZpZG9yLgogICAgRWwgY2xpZW50ZSBwdWVkZSBsZWVyIGVzdGUgcmVzb3VyY2UgcGFyYSBzYWJlciBxdcOpIGRvY3VtZW50b3MgZXhpc3RlbgogICAgYW50ZXMgZGUgcGVkaXIgdW5vIGNvbmNyZXRvLgogICAgIiIiCiAgICBwYXR0ZXJuID0gb3MucGF0aC5qb2luKERPQ1NfUEFUSCwgIioubWQiKQogICAgZmlsZXMgICA9IHNvcnRlZChnbG9iLmdsb2IocGF0dGVybikpCgogICAgaWYgbm90IGZpbGVzOgogICAgICAgIHJldHVybiAiTm8gaGF5IGRvY3VtZW50b3MgZGlzcG9uaWJsZXMuIgoKICAgIG5hbWVzID0gW29zLnBhdGguYmFzZW5hbWUoZikgZm9yIGYgaW4gZmlsZXNdCiAgICByZXR1cm4gIlxuIi5qb2luKG5hbWVzKQoKCkBtY3AucmVzb3VyY2UoImRvY3M6Ly97ZmlsZW5hbWV9IikKZGVmIHJlYWRfZG9jdW1lbnQoZmlsZW5hbWU6IHN0cikgLT4gc3RyOgogICAgIiIiCiAgICBMZWUgZWwgY29udGVuaWRvIGNvbXBsZXRvIGRlIHVuIGRvY3VtZW50by4KICAgIEVsIGNsaWVudGUgY29uc3RydXllIGxhIFVSSSBjb24gZWwgbm9tYnJlIGRlbCBmaWNoZXJvOgogICAgICAgIGRvY3M6Ly9hdXRoZW50aWNhdGlvbi5tZAogICAgICAgIGRvY3M6Ly9kYXRhYmFzZS5tZAogICAgICAgIGRvY3M6Ly9hcGlfZ2F0ZXdheS5tZAogICAgIiIiCiAgICAjIEV2aXRhciBwYXRoIHRyYXZlcnNhbDogZWwgZmljaGVybyBzb2xvIHB1ZWRlIGVzdGFyIGVuIERPQ1NfUEFUSAogICAgc2FmZV9uYW1lID0gb3MucGF0aC5iYXNlbmFtZShmaWxlbmFtZSkKICAgIGZpbGVwYXRoICA9IG9zLnBhdGguam9pbihET0NTX1BBVEgsIHNhZmVfbmFtZSkKCiAgICBpZiBub3Qgb3MucGF0aC5leGlzdHMoZmlsZXBhdGgpOgogICAgICAgIHJldHVybiBmIkRvY3VtZW50byAne3NhZmVfbmFtZX0nIG5vIGVuY29udHJhZG8uIFVzYSBkb2NzOi8vbGlzdCBwYXJhIHZlciBsb3MgZGlzcG9uaWJsZXMuIgoKICAgIHdpdGggb3BlbihmaWxlcGF0aCwgInIiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIHJldHVybiBmLnJlYWQoKQoKCiMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiMgVG9vbHMg4oCUIGFjY2lvbmVzIHF1ZSBlbCBjbGllbnRlIHB1ZWRlIGVqZWN1dGFyCiMKIyBVbmEgdG9vbCBlcyB1bmEgb3BlcmFjacOzbiwgY29tbyB1biBQT1NUIGVuIFJFU1QuCiMgRWwgY2xpZW50ZSAobyBlbCBMTE0pIGRlY2lkZSBjdcOhbmRvIGludm9jYXJsYSB5IGNvbiBxdcOpIHBhcsOhbWV0cm9zLgojIEZhc3RNQ1AgZ2VuZXJhIGVsIEpTT04gc2NoZW1hIGRlIGNhZGEgdG9vbCBhIHBhcnRpciBkZSBsb3MgdHlwZSBoaW50cy4KIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKCkBtY3AudG9vbCgpCmRlZiBzZWFyY2hfZG9jcyhxdWVyeTogc3RyKSAtPiBzdHI6CiAgICAiIiIKICAgIEJ1c2NhIHVuIHTDqXJtaW5vIGVuIHRvZGEgbGEgZG9jdW1lbnRhY2nDs24geSBkZXZ1ZWx2ZSBsb3MgZnJhZ21lbnRvcwogICAgcmVsZXZhbnRlcyBjb24gZWwgbm9tYnJlIGRlbCBkb2N1bWVudG8gZGUgb3JpZ2VuLgoKICAgIMOadGlsIGN1YW5kbyBubyBzYWJlcyBlbiBxdcOpIGRvY3VtZW50byBlc3TDoSBsYSBpbmZvcm1hY2nDs24g4oCUIGVsIExMTQogICAgcHVlZGUgYnVzY2FyIHByaW1lcm8geSBsdWVnbyBsZWVyIGVsIGRvY3VtZW50byBjb21wbGV0byBzaSBuZWNlc2l0YSBtw6FzLgogICAgIiIiCiAgICBwYXR0ZXJuID0gb3MucGF0aC5qb2luKERPQ1NfUEFUSCwgIioubWQiKQogICAgZmlsZXMgICA9IHNvcnRlZChnbG9iLmdsb2IocGF0dGVybikpCiAgICByZXN1bHRzID0gW10KCiAgICBxdWVyeV9sb3dlciA9IHF1ZXJ5Lmxvd2VyKCkKCiAgICBmb3IgZmlsZXBhdGggaW4gZmlsZXM6CiAgICAgICAgZmlsZW5hbWUgPSBvcy5wYXRoLmJhc2VuYW1lKGZpbGVwYXRoKQogICAgICAgIHdpdGggb3BlbihmaWxlcGF0aCwgInIiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgICAgICBsaW5lcyA9IGYucmVhZGxpbmVzKCkKCiAgICAgICAgZm9yIGksIGxpbmUgaW4gZW51bWVyYXRlKGxpbmVzKToKICAgICAgICAgICAgaWYgcXVlcnlfbG93ZXIgaW4gbGluZS5sb3dlcigpOgogICAgICAgICAgICAgICAgIyBEZXZvbHZlbW9zIGxhIGzDrW5lYSBjb24gYWxnbyBkZSBjb250ZXh0byAowrExIGzDrW5lYSkKICAgICAgICAgICAgICAgIHN0YXJ0ICAgPSBtYXgoMCwgaSAtIDEpCiAgICAgICAgICAgICAgICBlbmQgICAgID0gbWluKGxlbihsaW5lcyksIGkgKyAyKQogICAgICAgICAgICAgICAgZXhjZXJwdCA9ICIiLmpvaW4obGluZXNbc3RhcnQ6ZW5kXSkuc3RyaXAoKQogICAgICAgICAgICAgICAgcmVzdWx0cy5hcHBlbmQoZiJbe2ZpbGVuYW1lfV1cbntleGNlcnB0fSIpCgogICAgaWYgbm90IHJlc3VsdHM6CiAgICAgICAgcmV0dXJuIGYiTm8gc2UgZW5jb250cmFyb24gcmVzdWx0YWRvcyBwYXJhICd7cXVlcnl9Jy4iCgogICAgcmV0dXJuICJcblxuLS0tXG5cbiIuam9pbihyZXN1bHRzKQoKCkBtY3AudG9vbCgpCmRlZiBjcmVhdGVfdGlja2V0X2RyYWZ0KHRpdGxlOiBzdHIsIGRlc2NyaXB0aW9uOiBzdHIsIHNldmVyaXR5OiBzdHIgPSAiUDMiKSAtPiBzdHI6CiAgICAiIiIKICAgIEdlbmVyYSB1biBib3JyYWRvciBkZSB0aWNrZXQgZGUgc29wb3J0ZSBlc3RydWN0dXJhZG8uCiAgICBEZXZ1ZWx2ZSBlbCB0aWNrZXQgZm9ybWF0ZWFkbyBsaXN0byBwYXJhIGNvcGlhciBhbCBzaXN0ZW1hIGRlIHRpY2tldHMuCgogICAgc2V2ZXJpdHk6IFAxIChjcsOtdGljbykgfCBQMiAoYWx0bykgfCBQMyAobWVkaW8pIHwgUDQgKGJham8pCiAgICAiIiIKICAgIHZhbGlkX3NldmVyaXRpZXMgPSB7IlAxIiwgIlAyIiwgIlAzIiwgIlA0In0KICAgIGlmIHNldmVyaXR5IG5vdCBpbiB2YWxpZF9zZXZlcml0aWVzOgogICAgICAgIHNldmVyaXR5ID0gIlAzIgoKICAgICMgRW4gcHJvZHVjY2nDs24gYXF1w60gaXLDrWEgdW5hIGxsYW1hZGEgcmVhbCBhIEppcmEsIExpbmVhciwgZXRjLgogICAgIyBFbCBzZXJ2aWRvciBNQ1AgYWJzdHJhZSBlc2EgaW50ZWdyYWNpw7NuIOKAlCBlbCBjbGllbnRlIG5vIHNhYmUKICAgICMgc2kgZXN0byBlc2NyaWJlIGVuIEppcmEsIGVuIHVuYSBiYXNlIGRlIGRhdG9zIG8gZW4gdW4gZmljaGVyby4KICAgIGRyYWZ0ID0gZiIiIgpCT1JSQURPUiBERSBUSUNLRVQKPT09PT09PT09PT09PT09PT09ClTDrXR1bG86ICAgICAge3RpdGxlfQpTZXZlcmlkYWQ6ICAge3NldmVyaXR5fQpFc3RhZG86ICAgICAgQWJpZXJ0bwpBc2lnbmFkbyBhOiAgU2luIGFzaWduYXIKCkRlc2NyaXBjacOzbjoKe2Rlc2NyaXB0aW9ufQoKUGFzb3Mgc2lndWllbnRlczoKLSBbIF0gVmVyaWZpY2FyIGVuIGVudG9ybm8gZGUgc3RhZ2luZwotIFsgXSBSZXZpc2FyIGxvZ3MgZGVsIHNlcnZpY2lvIGFmZWN0YWRvCi0gWyBdIEVzY2FsYXIgc2kgbm8gaGF5IHJlc29sdWNpw7NuIGVuIHtfc2xhKHNldmVyaXR5KX0KIiIiLnN0cmlwKCkKCiAgICByZXR1cm4gZHJhZnQKCgpkZWYgX3NsYShzZXZlcml0eTogc3RyKSAtPiBzdHI6CiAgICAiIiJUaWVtcG8gbcOheGltbyBkZSByZXNvbHVjacOzbiBzZWfDum4gc2V2ZXJpZGFkLiIiIgogICAgcmV0dXJuIHsiUDEiOiAiMSBob3JhIiwgIlAyIjogIjQgaG9yYXMiLCAiUDMiOiAiMjQgaG9yYXMiLCAiUDQiOiAiNzIgaG9yYXMifS5nZXQoc2V2ZXJpdHksICIyNCBob3JhcyIpCgoKIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKIyBUb29scyDigJQgYmFzZSBkZSBkYXRvcwojIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKQG1jcC50b29sKCkKZGVmIGdldF9vcGVuX3RpY2tldHMoc2VydmljZTogc3RyID0gIiIsIHNldmVyaXR5OiBzdHIgPSAiIikgLT4gc3RyOgogICAgIiIiCiAgICBEZXZ1ZWx2ZSBsb3MgdGlja2V0cyBhYmllcnRvcyBkZSBsYSBiYXNlIGRlIGRhdG9zLgogICAgU2UgcHVlZGUgZmlsdHJhciBwb3Igc2VydmljaW8gKGF1dGgsIGRhdGFiYXNlLCBhcGlfZ2F0ZXdheSwgZnJvbnRlbmQpCiAgICB5L28gcG9yIHNldmVyaWRhZCAoUDEsIFAyLCBQMywgUDQpLgogICAgRGVqYXIgdmFjw61vIHBhcmEgdmVyIHRvZG9zIGxvcyB0aWNrZXRzIGFiaWVydG9zLgogICAgIiIiCiAgICBjb25uICA9IGdldF9kYigpCiAgICBxdWVyeSA9ICJTRUxFQ1QgKiBGUk9NIHRpY2tldHMgV0hFUkUgc3RhdHVzID0gJ29wZW4nIgogICAgcGFyYW1zOiBsaXN0ID0gW10KCiAgICBpZiBzZXJ2aWNlOgogICAgICAgIHF1ZXJ5ICs9ICIgQU5EIHNlcnZpY2UgPSA/IgogICAgICAgIHBhcmFtcy5hcHBlbmQoc2VydmljZSkKICAgIGlmIHNldmVyaXR5OgogICAgICAgIHF1ZXJ5ICs9ICIgQU5EIHNldmVyaXR5ID0gPyIKICAgICAgICBwYXJhbXMuYXBwZW5kKHNldmVyaXR5KQoKICAgIHF1ZXJ5ICs9ICIgT1JERVIgQlkgc2V2ZXJpdHksIGNyZWF0ZWRfYXQiCgogICAgcm93cyA9IGNvbm4uZXhlY3V0ZShxdWVyeSwgcGFyYW1zKS5mZXRjaGFsbCgpCiAgICBjb25uLmNsb3NlKCkKCiAgICBpZiBub3Qgcm93czoKICAgICAgICByZXR1cm4gIk5vIGhheSB0aWNrZXRzIGFiaWVydG9zIGNvbiBlc29zIGZpbHRyb3MuIgoKICAgIGxpbmVzID0gW10KICAgIGZvciByb3cgaW4gcm93czoKICAgICAgICBsaW5lcy5hcHBlbmQoCiAgICAgICAgICAgIGYiWyN7cm93WydpZCddfV0ge3Jvd1snc2V2ZXJpdHknXX0gfCB7cm93WydzZXJ2aWNlJ10gb3IgJ04vQSd9IHwge3Jvd1sndGl0bGUnXX1cbiIKICAgICAgICAgICAgZiIgIHtyb3dbJ2Rlc2NyaXB0aW9uJ11bOjEwMF19Li4uIgogICAgICAgICkKICAgIHJldHVybiAiXG5cbiIuam9pbihsaW5lcykKCgpAbWNwLnRvb2woKQpkZWYgc2F2ZV90aWNrZXQodGl0bGU6IHN0ciwgZGVzY3JpcHRpb246IHN0ciwgc2V2ZXJpdHk6IHN0ciA9ICJQMyIsIHNlcnZpY2U6IHN0ciA9ICIiKSAtPiBzdHI6CiAgICAiIiIKICAgIEd1YXJkYSB1biBudWV2byB0aWNrZXQgZW4gbGEgYmFzZSBkZSBkYXRvcyB5IGRldnVlbHZlIHN1IElELgogICAgQSBkaWZlcmVuY2lhIGRlIGNyZWF0ZV90aWNrZXRfZHJhZnQgKHF1ZSBzb2xvIGdlbmVyYSB0ZXh0byksCiAgICBlc3RhIHRvb2wgcGVyc2lzdGUgZWwgdGlja2V0IHBhcmEgcXVlIG90cm9zIHB1ZWRhbiBjb25zdWx0YXJsby4KCiAgICBzZXZlcml0eTogUDEgfCBQMiB8IFAzIHwgUDQKICAgIHNlcnZpY2U6ICBhdXRoIHwgZGF0YWJhc2UgfCBhcGlfZ2F0ZXdheSB8IGZyb250ZW5kIHwgKHZhY8OtbyBzaSBubyBhcGxpY2EpCiAgICAiIiIKICAgIHZhbGlkX3NldmVyaXRpZXMgPSB7IlAxIiwgIlAyIiwgIlAzIiwgIlA0In0KICAgIGlmIHNldmVyaXR5IG5vdCBpbiB2YWxpZF9zZXZlcml0aWVzOgogICAgICAgIHNldmVyaXR5ID0gIlAzIgoKICAgIGNvbm4gPSBnZXRfZGIoKQogICAgY3Vyc29yID0gY29ubi5leGVjdXRlKAogICAgICAgICJJTlNFUlQgSU5UTyB0aWNrZXRzICh0aXRsZSwgZGVzY3JpcHRpb24sIHNldmVyaXR5LCBzZXJ2aWNlLCBjcmVhdGVkX2F0KSBWQUxVRVMgKD8sPyw/LD8sPykiLAogICAgICAgICh0aXRsZSwgZGVzY3JpcHRpb24sIHNldmVyaXR5LCBzZXJ2aWNlIG9yIE5vbmUsIGRhdGV0aW1lLnV0Y25vdygpLmlzb2Zvcm1hdCgpKSwKICAgICkKICAgIHRpY2tldF9pZCA9IGN1cnNvci5sYXN0cm93aWQKICAgIGNvbm4uY29tbWl0KCkKICAgIGNvbm4uY2xvc2UoKQoKICAgIHJldHVybiBmIlRpY2tldCAje3RpY2tldF9pZH0gY3JlYWRvIGNvcnJlY3RhbWVudGUuXG5Uw610dWxvOiB7dGl0bGV9XG5TZXZlcmlkYWQ6IHtzZXZlcml0eX0g4oCUIFNMQToge19zbGEoc2V2ZXJpdHkpfSIKCgpAbWNwLnRvb2woKQpkZWYgY2xvc2VfdGlja2V0KHRpY2tldF9pZDogaW50LCByZXNvbHV0aW9uOiBzdHIpIC0+IHN0cjoKICAgICIiIgogICAgQ2llcnJhIHVuIHRpY2tldCBleGlzdGVudGUgY29uIGxhIHJlc29sdWNpw7NuIGFwbGljYWRhLgogICAgRGV2dWVsdmUgZXJyb3Igc2kgZWwgdGlja2V0IG5vIGV4aXN0ZSBvIHlhIGVzdMOhIGNlcnJhZG8uCiAgICAiIiIKICAgIGNvbm4gPSBnZXRfZGIoKQogICAgcm93ICA9IGNvbm4uZXhlY3V0ZSgiU0VMRUNUIHN0YXR1cyBGUk9NIHRpY2tldHMgV0hFUkUgaWQgPSA/IiwgKHRpY2tldF9pZCwpKS5mZXRjaG9uZSgpCgogICAgaWYgbm90IHJvdzoKICAgICAgICBjb25uLmNsb3NlKCkKICAgICAgICByZXR1cm4gZiJUaWNrZXQgI3t0aWNrZXRfaWR9IG5vIGVuY29udHJhZG8uIgoKICAgIGlmIHJvd1sic3RhdHVzIl0gPT0gImNsb3NlZCI6CiAgICAgICAgY29ubi5jbG9zZSgpCiAgICAgICAgcmV0dXJuIGYiVGlja2V0ICN7dGlja2V0X2lkfSB5YSBlc3TDoSBjZXJyYWRvLiIKCiAgICBjb25uLmV4ZWN1dGUoCiAgICAgICAgIlVQREFURSB0aWNrZXRzIFNFVCBzdGF0dXMgPSAnY2xvc2VkJywgcmVzb2x1dGlvbiA9ID8sIGNsb3NlZF9hdCA9ID8gV0hFUkUgaWQgPSA/IiwKICAgICAgICAocmVzb2x1dGlvbiwgZGF0ZXRpbWUudXRjbm93KCkuaXNvZm9ybWF0KCksIHRpY2tldF9pZCksCiAgICApCiAgICBjb25uLmNvbW1pdCgpCiAgICBjb25uLmNsb3NlKCkKCiAgICByZXR1cm4gZiJUaWNrZXQgI3t0aWNrZXRfaWR9IGNlcnJhZG8uXG5SZXNvbHVjacOzbjoge3Jlc29sdXRpb259IgoKCiMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiMgQXJyYW5xdWUKIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKCiMgSW5pY2lhbGl6YXIgbGEgQkQgYW50ZXMgZGUgcXVlIGVsIHNlcnZpZG9yIGVtcGllY2UgYSBhY2VwdGFyIGNvbmV4aW9uZXMuCiMgU2kgdGlja2V0cy5kYiBubyBleGlzdGUsIHNlIGNyZWEgYXF1w60gY29uIGxvcyBkYXRvcyBkZSBlamVtcGxvLgppbml0X2RiKCkKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICAjIG1jcC5ydW4oKSBhcnJhbmNhIGVsIHNlcnZpZG9yIGNvbiB0cmFuc3BvcnRlIHN0ZGlvLgogICAgIyBCbG9xdWVhIGhhc3RhIHF1ZSBlbCBjbGllbnRlIGNpZXJyYSBsYSBjb25leGnDs24uCiAgICAjIE5vIGhheSBvdXRwdXQgZW4gY29uc29sYSDigJQgbGEgY29tdW5pY2FjacOzbiBlcyBwb3Igc3RkaW4vc3Rkb3V0CiAgICAjIGVuIGZvcm1hdG8gSlNPTi1SUEMsIHF1ZSBlcyBlbCBwcm90b2NvbG8gcXVlIGhheSBkZWJham8gZGUgTUNQLgogICAgbWNwLnJ1bigpCg=='

SERVER_CODE = base64.b64decode(SERVER_CODE_B64).decode("utf-8")

with open("server.py", "w", encoding="utf-8") as f:
    f.write(SERVER_CODE)

print("✓ server.py escrito a disco — el cliente ya puede lanzarlo como subproceso")

---
# PARTE 2 — El cliente (`client.py`)

El cliente tiene **dos responsabilidades**:

1. **Hablar con el servidor MCP** — conectar, descubrir capacidades, ejecutar tools y leer resources
2. **Actuar como puente entre Claude y el servidor** — el LLM decide qué tools invocar, el cliente las ejecuta en el servidor y devuelve los resultados

> La SDK de MCP es **asíncrona** (asyncio), por eso todas las funciones que hablan con el servidor son `async/await`.

## Setup del cliente

`StdioServerParameters` le dice al cliente cómo arrancar el servidor: lanza `python server.py` como subproceso. **No hace falta arrancar el servidor a mano** — el cliente lo gestiona.

> **Nota:** usamos `sys.executable` en vez de `"python"` para garantizar que el subproceso se ejecuta con el mismo Python que el notebook (en macOS muchas veces solo existe `python3`, no `python`).

In [ ]:
"""
L7 — MCP Client: se conecta al servidor y usa Claude como agente
================================================================
El cliente tiene dos responsabilidades:

    1. Hablar con el servidor MCP — conectar, descubrir capacidades,
       ejecutar tools y leer resources.

    2. Actuar como puente entre Claude y el servidor — el LLM decide
       qué tools invocar, el cliente las ejecuta en el servidor y
       devuelve los resultados.

La SDK de MCP es asíncrona (asyncio), por eso todas las funciones
que hablan con el servidor son async/await.

Requisitos:
    pip install anthropic mcp
    export ANTHROPIC_API_KEY="sk-ant-..."

Uso:
    python client.py
    (arranca server.py automáticamente como subproceso)
"""

from pathlib import Path
from dotenv import load_dotenv
for _p in [Path.cwd()] + list(Path.cwd().parents):
    if (_p / ".env").exists():
        load_dotenv(_p / ".env", override=True)
        break
import asyncio
import json
import sys
import anthropic
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


MODEL = "claude-haiku-4-5-20251001"

# StdioServerParameters le dice al cliente cómo arrancar el servidor.
# El cliente lanza `python server.py` como subproceso y se comunica
# con él por stdin/stdout — no hace falta arrancar el servidor a mano.
SERVER_PARAMS = StdioServerParameters(
    command=sys.executable,  # mismo Python que el notebook (evita problemas de PATH)
    args=["server.py"],
)

## Discovery — descubrir qué expone el servidor

Esta función ilustra **el protocolo en sí**: conectamos al servidor y listamos sus capacidades **sin invocar ninguna lógica de LLM**. El cliente no sabe nada de antemano — todo lo descubre al conectar.

In [ ]:
# ─────────────────────────────────────────────
# Parte 1 — Discovery: qué expone el servidor
#
# Conectamos al servidor y listamos sus capacidades sin invocar
# ninguna lógica de LLM. Esto muestra el protocolo en sí:
# el cliente no sabe nada de antemano — todo lo descubre al conectar.
# ─────────────────────────────────────────────

async def discover(session: ClientSession) -> None:
    """Muestra qué resources y tools expone el servidor."""

    print("\n[Discovery] Resources disponibles:")
    resources = await session.list_resources()
    for r in resources.resources:
        print(f"  • {r.uri}  —  {r.description or r.name}")

    print("\n[Discovery] Tools disponibles:")
    tools = await session.list_tools()
    for t in tools.tools:
        params = list(t.inputSchema.get("properties", {}).keys())
        print(f"  • {t.name}({', '.join(params)})  —  {t.description}")

## Claude + MCP — el puente entre el modelo y el servidor

Dos piezas:

- **`mcp_tools_to_anthropic`**: convierte las tools del formato MCP (`inputSchema`) al formato que espera la API de Anthropic (`input_schema`). Mismo contenido, distinto nombre de campo.

- **`ask`**: el bucle principal. Es **idéntico al de L3**, con una diferencia clave: las tools no están hardcodeadas — vienen del servidor MCP via `session.list_tools()`. Si mañana añades una tool nueva al servidor, este código no cambia.

In [ ]:
# ─────────────────────────────────────────────
# Parte 2 — Claude + MCP
#
# Las tools que descubrió el cliente se pasan a Claude.
# Cuando Claude decide invocar una, el cliente la ejecuta
# en el servidor MCP y devuelve el resultado.
# ─────────────────────────────────────────────

def mcp_tools_to_anthropic(tools) -> list[dict]:
    """
    Convierte las tools del formato MCP al formato que espera la API de Anthropic.
    MCP usa inputSchema (JSON Schema), Anthropic usa input_schema — mismo contenido,
    distinto nombre de campo.
    """
    return [
        {
            "name":         tool.name,
            "description":  tool.description or "",
            "input_schema": tool.inputSchema,
        }
        for tool in tools
    ]


async def ask(session: ClientSession, question: str) -> str:
    """
    Responde una pregunta usando Claude con acceso al servidor MCP.

    Flujo:
        1. Descubre las tools del servidor
        2. Pasa la pregunta + tools a Claude
        3. Si Claude llama una tool → la ejecuta en el servidor
        4. Devuelve el resultado al modelo y continúa
        5. Cuando Claude termina → devuelve la respuesta final
    """
    print(f"\n{'=' * 60}")
    print(f"PREGUNTA: {question}")
    print(f"{'=' * 60}")

    # Descubrir tools en cada llamada garantiza que siempre usamos
    # las capacidades actuales del servidor, aunque cambien en caliente
    tools_result     = await session.list_tools()
    anthropic_tools  = mcp_tools_to_anthropic(tools_result.tools)

    client   = anthropic.Anthropic()
    messages = [{"role": "user", "content": question}]

    # Bucle idéntico a L3 — la diferencia es que las tools no están
    # hardcodeadas en este fichero: vienen del servidor MCP
    while True:
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            temperature=0,
            tools=anthropic_tools,
            messages=messages,
        )

        print(f"\n[Modelo] stop_reason: {response.stop_reason}")

        if response.stop_reason == "end_turn":
            answer = next(b.text for b in response.content if b.type == "text")
            print(f"\n[Respuesta]\n{answer}")
            return answer

        if response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})

            tool_results = []
            for block in response.content:
                if block.type != "tool_use":
                    continue

                print(f"\n[Tool] {block.name}({json.dumps(block.input, ensure_ascii=False)})")

                # El cliente ejecuta la tool en el servidor MCP.
                # Si mañana cambias la implementación de search_docs en server.py,
                # este código no cambia — solo cambia el servidor.
                result = await session.call_tool(block.name, block.input)

                # result.content es una lista de objetos de contenido;
                # para tools de texto tomamos el primero
                tool_output = result.content[0].text if result.content else ""
                print(f"[Tool result] {tool_output[:120]}...")

                tool_results.append({
                    "type":        "tool_result",
                    "tool_use_id": block.id,
                    "content":     tool_output,
                })

            messages.append({"role": "user", "content": tool_results})

## `main` — pone todo junto

`stdio_client(SERVER_PARAMS)` arranca `server.py` como subproceso y gestiona la conexión. Al salir del bloque `async with`, el subproceso se termina limpiamente.

Las preguntas de ejemplo cubren los tres tipos de operaciones:
- **Búsqueda en docs** (lectura via tool)
- **Consulta a la BD** (lectura via tool)
- **Cierre de ticket** (escritura via tool)
- **Creación de ticket** (escritura via tool)

In [ ]:
# ─────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────

async def main():
    # stdio_client arranca server.py como subproceso y gestiona
    # la conexión. Al salir del bloque `async with`, el subproceso
    # se termina limpiamente.
    async with stdio_client(SERVER_PARAMS) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # ── Parte 1: ver qué expone el servidor ──
            print("=" * 60)
            print("DISCOVERY — capacidades del servidor MCP")
            print("=" * 60)
            await discover(session)

            # ── Parte 2: Claude usando el servidor ──
            print("\n\n" + "=" * 60)
            print("AGENTE — Claude con acceso al servidor MCP")
            print("=" * 60)

            questions = [
                # Busca en docs
                "¿Qué hago si el servicio de autenticación devuelve errores 500?",

                # Consulta la BD
                "¿Qué tickets críticos tenemos abiertos ahora mismo?",

                # Combina docs + BD: busca cómo resolverlo y cierra el ticket
                "El ticket #3 era el circuit breaker del API gateway. Ya lo hemos resuelto reiniciando Kong. Ciérralo.",

                # Crea ticket nuevo en la BD
                "Abre un ticket P2 para el servicio de base de datos: estamos viendo queries lentas en producción, "
                "algunas tardan más de 30 segundos.",
            ]

            for question in questions:
                await ask(session, question)
                print()

## Ejecutar — el cliente conectándose al servidor

> **Asegúrate** de haber ejecutado la celda que escribe `server.py` a disco (más arriba en este notebook). Sin ese paso el subproceso fallará con "file not found".

> **En Jupyter** usamos `await main()` directamente (top-level await — soportado en IPython 7+). Verás:
> 1. La fase de **DISCOVERY** — qué expone el servidor
> 2. **Claude usando el servidor** — invocando tools para responder cada pregunta

In [ ]:
await main()

## MCP en el ecosistema real

MCP fue publicado por Anthropic en noviembre de 2024 y la adopción ha sido muy rápida. Hoy hay servidores MCP públicos para:

- **Repos**: GitHub, GitLab
- **Tickets**: Jira, Linear
- **Bases de datos**: PostgreSQL, SQLite
- **Comunicación**: Slack, Notion
- **Infra**: Docker, Kubernetes

Esto significa que en lugar de construir la integración desde cero, **conectas un servidor existente y el modelo tiene acceso inmediato a esos sistemas**.

## Has llegado al final 🎉

Has recorrido la escalera completa de workflows de IA:

| Nivel | Lo que aprendiste |
|-------|-------------------|
| L1 | Qué es una llamada al LLM y los conceptos básicos (tokens, temperature, system/user) |
| L2 | Encadenar llamadas con flujo determinista controlado por código |
| L3 | Dar al modelo herramientas externas que puede invocar |
| L4 | RAG — recuperar contexto relevante de una base de conocimiento |
| L5 | Agentes — el modelo persigue un objetivo de forma autónoma |
| L6 | Multi-agente — orquestador + especialistas en paralelo |
| L7 | MCP — protocolo estándar para integrar cualquier sistema |

> **El repo se irá actualizando a medida que avance en el aprendizaje.**